# Multi-environment config selection

A single dw.json can carry a `configs` array of **named** environments. This
notebook reads the bundled `../multi-env.example.json` (pseudo values, no
network needed) and shows how `ResolveConfigOptions(instance=...)` picks one.

**Selection priority** (same as the `b2c` CLI):
1. the requested `instance` name, else
2. the config marked `active: true`, else
3. the root-level config.

Each named config is **self-contained** — root-level keys are *not* merged into
a selected named entry, so give every environment its own full credentials.

In [ ]:
from pathlib import Path

from b2c_tooling_sdk import ResolveConfigOptions, resolve_config

MULTI_ENV = Path("../multi-env.example.json").resolve()
print("Reading", MULTI_ENV.name)

In [ ]:
for selection in [None, "staging", "production", "does-not-exist"]:
    config = await resolve_config(
        options=ResolveConfigOptions(config_path=str(MULTI_ENV), instance=selection)
    )
    values = config.values
    label = selection if selection is not None else "(default → active/root)"
    print(f"instance={label!r}")
    print(f"    hostname : {values.hostname}")
    print(f"    clientId : {values.client_id}")
    print()

## Using a selection

Once a selection carries real credentials, build an instance and call APIs just
like the other notebooks:

```python
config = await resolve_config(options=ResolveConfigOptions(instance="production"))
instance = config.create_b2c_instance()
```

You can also point `config_path` at your own multi-config dw.json, or manage
named entries with `add_instance` / `remove_instance` / `set_active_instance`
from `b2c_tooling_sdk.config`.